# Hackathon 9 v3: Vision Transformers y stacking

Tres clasificadores basados en ViT preentrenados en ImageNet se adaptan a seis clases. Se usa el mismo ZIP y cinco folds externos estratificados por grupos de imágenes idénticas (semilla 42). Cada fold externo se divide en 64 % entrenamiento, 16 % validación/meta y 20 % prueba. El fold de prueba nunca participa en el ajuste ni la selección de época. Precision y F1 son macro; la desviación estándar es muestral entre los cinco folds. El objetivo de precisión > 0.95 es una meta, no un resultado garantizado.


### Configuración e imports

In [ ]:
import os, json, random, time, traceback, copy, zipfile, gc, hashlib
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
import torch, timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT=Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
progress={'status':'running','stage':'setup','device':str(DEVICE),'artifacts':[]}
def save(): (OUT/'run_summary.json').write_text(json.dumps(progress,indent=2,ensure_ascii=False),encoding='utf-8')
def stage(name, fn):
    if progress['status']=='failed': return
    progress['stage']=name; save()
    try: fn()
    except Exception as e:
        progress.update(status='failed',error=repr(e),traceback=traceback.format_exc()); save(); print(progress['traceback'])
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU','timm:',timm.__version__)
save()


### Datos y exploración
Se comprueba el balance y se muestran ejemplos. El particionado estratificado conserva proporciones por clase y utiliza cada imagen una vez como prueba externa.

In [ ]:
CLASSES=['chicken_wings','hamburger','hot_dog','pizza','sushi','tacos']
def inspect_data():
    global frame
    assert DEVICE.type=='cuda', 'Se requiere GPU'
    root=Path('/kaggle/input'); archives=list(root.rglob('fast_food.zip')); dest=Path('/tmp/hack9_images')
    if archives:
        with zipfile.ZipFile(archives[0]) as z: z.extractall(dest)
    else: dest=root
    paths=[]
    for label in CLASSES:
        dirs=[x for x in dest.rglob(label) if x.is_dir()]; assert len(dirs)==1,(label,dirs)
        paths.extend((str(f),label) for f in sorted(dirs[0].glob('*')) if f.suffix.lower() in ('.jpg','.jpeg','.png'))
    frame=pd.DataFrame(paths,columns=['path','label']); frame['target']=frame.label.map({x:i for i,x in enumerate(CLASSES)})
    assert len(frame)==6000 and frame.groupby('label').size().min()>=5
    frame['group']=frame.path.map(lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest())
    assert frame.groupby('group').target.nunique().max()==1, 'Duplicados con etiquetas distintas'
    print('Grupos duplicados exactos:',int((frame.groupby('group').size()>1).sum()))
    print('Imágenes:',len(frame)); print(frame.groupby('label').size().to_string())
    frame.groupby('label').size().to_csv(OUT/'balance.csv',header=['imagenes'])
    fig,axes=plt.subplots(1,6,figsize=(16,3))
    for ax,label in zip(axes,CLASSES):
        with Image.open(frame[frame.label==label].path.iloc[0]) as img: ax.imshow(img.convert('RGB'))
        ax.set_title(label); ax.axis('off')
    fig.tight_layout(); fig.savefig(OUT/'ejemplos.png',dpi=120); plt.show()
stage('datos',inspect_data)


### Preprocesamiento y modelos
El recorte aleatorio, el volteo y los cambios leves de color aportan variación de escala, orientación e iluminación. Las imágenes de validación y prueba reciben únicamente redimensionamiento y recorte central. Cada modelo aplica la media y desviación especificadas por sus pesos. Se entrenan la cabeza y los últimos cuatro bloques; el resto permanece congelado para reducir tiempo.

In [ ]:
MODELS={
 'ViT-Small':'vit_small_patch16_224.augreg_in21k_ft_in1k',
 'DeiT3-Small':'deit3_small_patch16_224.fb_in22k_ft_in1k',
 'ViT-Base':'vit_base_patch16_224.augreg_in21k_ft_in1k',
}
EPOCHS=8; BATCH_SIZE=32
class Food(Dataset):
    def __init__(self,rows,tf): self.rows=rows.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]
        with Image.open(row.path) as img: x=self.tf(img.convert('RGB'))
        return x,int(row.target)
def make_transforms(model):
    cfg=timm.data.resolve_model_data_config(model); mean=cfg['mean']; std=cfg['std']
    train=transforms.Compose([transforms.RandomResizedCrop(224,scale=(0.75,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(0.15,0.15,0.15,0.05),transforms.ToTensor(),transforms.Normalize(mean,std)])
    valid=transforms.Compose([transforms.Resize(256),transforms.CenterCrop(224),transforms.ToTensor(),transforms.Normalize(mean,std)])
    return train,valid
def loader(indices,tf,training=False):
    return DataLoader(Food(frame.iloc[indices],tf),batch_size=BATCH_SIZE,shuffle=training,num_workers=2,pin_memory=True)
def create_model(arch):
    model=timm.create_model(arch,pretrained=True,num_classes=6)
    for p in model.parameters(): p.requires_grad=False
    for block in model.blocks[-4:]:
        for p in block.parameters(): p.requires_grad=True
    for module in (model.norm,model.get_classifier()):
        for p in module.parameters(): p.requires_grad=True
    return model.to(DEVICE)


### Ajuste y predicción
AdamW usa tasas diferenciadas para la cabeza y el backbone. Se selecciona la época con mejor F1 macro en la partición interna; esta misma partición sirve para entrenar el meta-modelo, por lo que la estimación válida de desempeño es siempre el fold externo. Se guarda el historial después de cada época.

In [ ]:
history=[]; metrics=[]; all_predictions=[]; meta_rows=[]
def train_model(name,arch,fold,train_idx,meta_idx,test_idx):
    torch.manual_seed(SEED+fold); torch.cuda.manual_seed_all(SEED+fold)
    model=create_model(arch); train_tf,valid_tf=make_transforms(model)
    head=list(model.get_classifier().parameters()); head_ids={id(p) for p in head}
    backbone=[p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    optimizer=torch.optim.AdamW([{'params':backbone,'lr':3e-5},{'params':head,'lr':8e-4}],weight_decay=1e-4)
    scaler=torch.amp.GradScaler('cuda'); loss_fn=nn.CrossEntropyLoss(); best=-1; best_state=None
    train_loader=loader(train_idx,train_tf,True); meta_loader=loader(meta_idx,valid_tf)
    def predict(data_loader,with_loss=False):
        model.eval(); probs=[]; ys=[]; losses=[]
        with torch.no_grad():
            for x,y in data_loader:
                x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
                with torch.autocast(device_type='cuda',dtype=torch.float16): logits=model(x); loss=loss_fn(logits,y)
                probs.append(logits.float().softmax(1).cpu().numpy()); ys.append(y.cpu().numpy()); losses.append((loss.item(),len(y)))
        return np.concatenate(probs),np.concatenate(ys),sum(a*n for a,n in losses)/sum(n for _,n in losses)
    for epoch in range(EPOCHS):
        model.train(); total=correct=loss_sum=0
        for x,y in train_loader:
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16): logits=model(x); loss=loss_fn(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total+=len(y); loss_sum+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item()
        probs,truth,val_loss=predict(meta_loader); val_f1=f1_score(truth,probs.argmax(1),average='macro',zero_division=0)
        if val_f1>best:
            best=val_f1; best_state=copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()}); best_epoch=epoch+1
        history.append(dict(fold=fold,model=name,epoch=epoch+1,train_loss=loss_sum/total,train_accuracy=correct/total,meta_loss=val_loss,meta_f1=val_f1))
        pd.DataFrame(history).to_csv(OUT/'historial.csv',index=False)
        print(f'{name} fold {fold} época {epoch+1}: train={loss_sum/total:.3f}, val_f1={val_f1:.4f}',flush=True)
    model.load_state_dict(best_state); del best_state
    a,_,_=predict(meta_loader); b,_,_=predict(loader(test_idx,valid_tf));
    print(f'{name} fold {fold}: mejor época {best_epoch}, F1 interno {best:.4f}',flush=True)
    del model,optimizer,train_loader,meta_loader; gc.collect(); torch.cuda.empty_cache()
    return a,b
def score(y,p,fold,name):
    metrics.append(dict(fold=fold,model=name,precision_macro=precision_score(y,p,average='macro',zero_division=0),f1_macro=f1_score(y,p,average='macro',zero_division=0)))


### Cinco folds y stacking
Los duplicados exactos se agrupan por SHA-256 y permanecen en la misma partición, incluso en la división interna. Cada modelo genera seis probabilidades para las imágenes internas y externas. Una regresión logística se ajusta solo con las 18 probabilidades internas. Se comparan los modelos, el promedio y el stacking sobre las mismas imágenes externas. La selección de época usa el conjunto meta, por lo que sus métricas internas no son una estimación independiente.

In [ ]:
def experiment():
    outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED); y=frame.target.to_numpy(); groups=frame.group.to_numpy(); split_rows=[]
    for fold,(outer_train,test_idx) in enumerate(outer.split(np.zeros(len(y)),y,groups),1):
        representatives=frame.iloc[outer_train].drop_duplicates('group')
        train_groups,meta_groups=train_test_split(representatives.group.to_numpy(),test_size=0.2,stratify=representatives.target.to_numpy(),random_state=SEED+fold)
        train_idx=outer_train[np.isin(groups[outer_train],train_groups)]; meta_idx=outer_train[np.isin(groups[outer_train],meta_groups)]
        assert not (set(groups[train_idx])&set(groups[meta_idx]) or set(groups[train_idx])&set(groups[test_idx]) or set(groups[meta_idx])&set(groups[test_idx]))
        for partition,indices in [('train',train_idx),('meta',meta_idx),('test',test_idx)]:
            split_rows.extend({'fold':fold,'partition':partition,'index':int(i),'group':groups[i]} for i in indices)
        pd.DataFrame(split_rows).to_csv(OUT/'particiones.csv',index=False)
        assert not (set(train_idx)&set(meta_idx) or set(train_idx)&set(test_idx) or set(meta_idx)&set(test_idx))
        meta_probs=[]; test_probs=[]
        for name,arch in MODELS.items():
            progress.update(stage='entrenamiento',fold=fold,model=name); save()
            a,b=train_model(name,arch,fold,train_idx,meta_idx,test_idx); meta_probs.append(a); test_probs.append(b)
            score(y[test_idx],b.argmax(1),fold,name)
        meta_x=np.concatenate(meta_probs,axis=1); test_x=np.concatenate(test_probs,axis=1)
        meta=LogisticRegression(max_iter=2000,random_state=SEED).fit(meta_x,y[meta_idx]); pred=meta.predict(test_x)
        score(y[test_idx],pred,fold,'Stacking'); avg=np.mean(test_probs,axis=0).argmax(1); score(y[test_idx],avg,fold,'Promedio')
        for i,idx in enumerate(test_idx):
            row={'fold':fold,'index':int(idx),'path':frame.path.iloc[idx],'truth':int(y[idx]),'group':groups[idx],'Stacking':int(pred[i]),'Promedio':int(avg[i])}
            row.update({name:int(probs[i].argmax()) for name,probs in zip(MODELS,test_probs)}); all_predictions.append(row)
        columns=[f'{name}_{label}' for name in MODELS for label in CLASSES]
        mf=pd.DataFrame(meta_x,columns=columns); mf.insert(0,'index',meta_idx); mf.insert(0,'target',y[meta_idx]); mf.insert(0,'fold',fold); meta_rows.append(mf)
        pd.DataFrame(metrics).to_csv(OUT/'metricas_por_fold.csv',index=False)
        pd.DataFrame(all_predictions).to_csv(OUT/'predicciones.csv',index=False)
        pd.concat(meta_rows,ignore_index=True).to_csv(OUT/'dataset_meta.csv',index=False)
        print('Fold terminado:',fold,flush=True)
stage('experimento',experiment)


### Resultados, matrices y curvas
La desviación entre folds describe variabilidad, no un intervalo de confianza. La diferencia entre stacking y el mejor modelo individual es descriptiva. El oráculo por parejas supone conocer qué modelo acierta y solo mide complementariedad potencial.

In [ ]:
def report():
    m=pd.DataFrame(metrics); summary=m.groupby('model')[['precision_macro','f1_macro']].agg(['mean','std']).sort_values(('f1_macro','mean'),ascending=False)
    summary.to_csv(OUT/'resumen_metricas.csv'); display(summary)
    pred=pd.DataFrame(all_predictions); assert len(pred)==6000 and pred['index'].nunique()==6000
    fig,axes=plt.subplots(1,5,figsize=(24,4))
    for ax,name in zip(axes,list(MODELS)+['Promedio','Stacking']):
        ConfusionMatrixDisplay(confusion_matrix(pred.truth,pred[name],labels=range(6)),display_labels=CLASSES).plot(ax=ax,colorbar=False,xticks_rotation=90); ax.set_title(name)
    fig.tight_layout(); fig.savefig(OUT/'matrices_confusion.png',dpi=130); plt.show()
    h=pd.DataFrame(history); fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,name in zip(axes,MODELS):
        h[h.model==name].groupby('epoch')[['train_loss','meta_loss']].mean().plot(ax=ax); ax.set_title(name); ax.set_ylabel('Pérdida media'); ax.set_xlabel('Época')
    fig.tight_layout(); fig.savefig(OUT/'curvas_entrenamiento.png',dpi=130); plt.show()
    best=summary.loc[list(MODELS)].sort_values(('f1_macro','mean'),ascending=False).index[0]
    cm=confusion_matrix(pred.truth,pred.Stacking,labels=range(6)); errors=cm.sum(axis=1)-np.diag(cm)
    complement={f'{a} + {b}':float(((pred[a]==pred.truth)|(pred[b]==pred.truth)).mean()) for i,a in enumerate(MODELS) for b in list(MODELS)[i+1:]}
    analysis={'mejor_individual':best,'diferencia_f1_stacking_menos_mejor':float(summary.loc['Stacking',('f1_macro','mean')]-summary.loc[best,('f1_macro','mean')]),'errores_stacking_por_clase':dict(zip(CLASSES,errors.tolist())),'acierto_oraculo_por_pareja':complement,'precision_objetivo_superado':bool((summary[('precision_macro','mean')]>0.95).any()),'nota':'El oráculo es un límite descriptivo, no un ensemble implementable.'}
    (OUT/'analisis.json').write_text(json.dumps(analysis,indent=2,ensure_ascii=False),encoding='utf-8'); print(json.dumps(analysis,indent=2,ensure_ascii=False))
    progress.update(status='complete',stage='done',artifacts=[x.name for x in OUT.iterdir() if x.is_file()]); save()
stage('informe',report)
print('Estado:',progress['status'],'etapa:',progress['stage'])
